In [2]:
# Parameters

In [3]:
from lsst.daf.butler import Butler
import datetime

In [4]:
butler = Butler('main', collections=['LSSTCam/raw/all', 'LSSTCam/calib/unbounded'])

dayObs = 20250415
instrument = "LSSTCam"

exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"""
                    exposure.day_obs>={dayObs} and
                    instrument='LSSTCam'
                    """
                                                   ):
    exposureList.append([record.id, record])

print(f"Image count in butler since {dayObs} by {datetime.datetime.now()} is {len(exposureList)}")
for atype in set([ exp[1].observation_type for exp in exposureList ]):
    print( f"N({atype:<10})= {len(list(filter( lambda x: x[1].observation_type==atype, exposureList ))):>10},", end=' ')
    print( f"({len(list(filter( lambda x: ( x[1].observation_type==atype ) and ( x[1].can_see_sky==True ), exposureList ))):>10})")
    
print(f"Number in the parensis is with the can_see_sky=True condition")


Image count in butler since 20250415 by 2026-09-02 17:04:15.627054 is 214585
N(flat      )=      36522, (       318)
N(science   )=      99271, (     99155)
N(test      )=        272, (         0)
N(acq       )=      35839, (     35635)
N(bias      )=      13205, (         4)
N(dark      )=      17059, (         0)
N(indome    )=         59, (         0)
N(focus     )=        158, (       158)
N(cbp       )=       2248, (         0)
N(unknown   )=        648, (         0)
N(stuttered )=          6, (         0)
N(engtest   )=       2066, (      2057)
N(cwfs      )=       7232, (      7225)
Number in the parensis is with the can_see_sky=True condition


In [5]:
import os
import pandas as pd
from lsst.summit.utils import ConsDbClient

DAY_OBS_MIN = 20250415          # April 2025 onward (day_obs is a YYYYMMDD int)
INSTRUMENT = 'lsstcam'          # 'latiss' for AuxTel

os.environ['no_proxy'] = os.environ.get('no_proxy', '') + ',.consdb'
cdb = ConsDbClient('http://consdb-pq.consdb:8080/consdb')   # in-pod (RSP)
# Off the RSP: ConsDbClient('https://user:<token>@usdf-rsp.slac.stanford.edu/consdb')

# One pass over the exposure table: img_type x shutter state. shut_time is the
# "spatially-averaged shutter-open duration", so shut_time > 0 is a direct
# physical test for an open shutter -- no need to enumerate calibration
# img_types. The NULL bucket is kept explicitly so shutter-open images that are
# merely missing the value don't vanish silently into the > 0 comparison.
query = f"""
    SELECT img_type,
           CASE WHEN shut_time IS NULL THEN 'null'
                WHEN shut_time > 0     THEN 'open'
                ELSE 'zero' END AS shutter,
           COUNT(*)       AS n,
           SUM(shut_time) AS shut_seconds,
           MIN(day_obs)   AS first_night,
           MAX(day_obs)   AS last_night
    FROM cdb_{INSTRUMENT}.exposure
    WHERE day_obs >= {DAY_OBS_MIN}
    GROUP BY 1, 2
"""
df = cdb.query(query).to_pandas()
for c in ('n', 'shut_seconds', 'first_night', 'last_night'):
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['img_type'] = df.img_type.fillna('(none)')

openm = df.shutter == 'open'
n_open = int(df.loc[openm, 'n'].sum())
hours = df.loc[openm, 'shut_seconds'].sum() / 3600.0

print(f'=== cdb_{INSTRUMENT}.exposure, day_obs >= {DAY_OBS_MIN} ===')
print(f'\nImages with the shutter open : {n_open:,}')
print(f'Total shutter-open time      : {hours:,.1f} h  ({hours / 24:,.1f} days)')
if openm.any():
    print(f'Night range                  : {int(df.loc[openm, "first_night"].min())} '
          f'.. {int(df.loc[openm, "last_night"].max())}')

# Audit: make sure "open" isn't losing images to a NULL shut_time
print('\n--- shutter state (all exposures in window) ---')
audit = df.groupby('shutter', as_index=False)['n'].sum().sort_values('n', ascending=False)
audit['pct'] = 100.0 * audit.n / audit.n.sum()
print(audit.to_string(index=False, float_format=lambda v: f'{v:.2f}'))
n_null = int(audit.loc[audit.shutter == 'null', 'n'].sum())
if n_null:
    print(f'\n  NOTE: {n_null:,} exposures have shut_time IS NULL and are NOT counted '
          f'above.\n  For those, fall back to img_type NOT IN (\'bias\', \'dark\'):')
    print(df[(df.shutter == 'null')].groupby('img_type', as_index=False)['n']
          .sum().sort_values('n', ascending=False).to_string(index=False))

# Breakdown of the shutter-open images
print('\n--- shutter-open images by img_type ---')
by_type = (df[openm].groupby('img_type', as_index=False)
           .agg(n=('n', 'sum'), shut_seconds=('shut_seconds', 'sum')))
by_type['hours'] = by_type.shut_seconds / 3600.0
by_type['pct'] = 100.0 * by_type.n / max(n_open, 1)
print(by_type[['img_type', 'n', 'hours', 'pct']]
      .sort_values('n', ascending=False)
      .to_string(index=False, float_format=lambda v: f'{v:.1f}'))

        Use pytest instead. [astropy.tests.runner]


        Use pytest instead. [astropy.utils.decorators]


=== cdb_lsstcam.exposure, day_obs >= 20250415 ===

Images with the shutter open : 200,614
Total shutter-open time      : 1,599.3 h  (66.6 days)
Night range                  : 20250415 .. 20260714

--- shutter state (all exposures in window) ---
shutter      n   pct
   open 200614 93.87
   zero  13090  6.13

--- shutter-open images by img_type ---
img_type     n  hours  pct
 science 99264  833.9 49.5
    flat 36364  253.9 18.1
     acq 36263  290.1 18.1
    dark 16920  136.6  8.4
    cwfs  7254   56.2  3.6
     cbp  2248   11.1  1.1
 engtest  2065   16.5  1.0
   focus   176    0.8  0.1
  indome    59    0.2  0.0
    test     1    0.0  0.0
